In [4]:
import os
import sys
import inspect
import subprocess

import numpy as np
import torch

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# ==============================================================================
# 1. NHẬN DIỆN MÔI TRƯỜNG (COLAB / LOCAL)
# ==============================================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IS_COLAB = True
    CHECKPOINT_DIR = "/content/drive/MyDrive/python-en-vi-mt/checkpoints_trial"
    print("Đang chạy trên Google Colab -> checkpoint lưu tại Google Drive.")
except ImportError:
    IS_COLAB = False
    CHECKPOINT_DIR = "checkpoints_trial"
    print("Đang chạy trên máy cá nhân -> checkpoint lưu tại thư mục dự án.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if IS_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "datasets", "evaluate", "sacrebleu",
        "sentencepiece", "sacremoses", "accelerate", "ipywidgets",
    ])

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from transformers.trainer_utils import get_last_checkpoint
import evaluate

# ==============================================================================
# 2. CẤU HÌNH
# ==============================================================================
MODEL_NAME = "Helsinki-NLP/opus-mt-en-vi"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 1
EVAL_STEPS = 500
NUM_BEAMS_TEST = 5
GEN_BATCH_SIZE = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""))
print(f"Thư mục checkpoint: {CHECKPOINT_DIR}")

# ==============================================================================
# 3. TẢI DỮ LIỆU IWSLT15 (PARQUET, TRÁNH SERVER STANFORD 403)
# ==============================================================================
print("\nĐang tải tập dữ liệu IWSLT15 en-vi...")

BASE = "hf://datasets/IWSLT/mt_eng_vietnamese@refs/convert/parquet/iwslt2015-en-vi"
try:
    raw_datasets = load_dataset(
        "parquet",
        data_files={
            "train": f"{BASE}/train/0000.parquet",
            "validation": f"{BASE}/validation/0000.parquet",
            "test": f"{BASE}/test/0000.parquet",
        },
    )
    print("Tải thành công từ nhánh parquet của IWSLT/mt_eng_vietnamese")
except Exception as e1:
    print(f"Tải parquet thất bại: {e1!r}")
    print("Thử dùng revision parquet qua load_dataset...")
    raw_datasets = load_dataset(
        "IWSLT/mt_eng_vietnamese", "iwslt2015-en-vi", revision="refs/convert/parquet"
    )

print(raw_datasets)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def extract_pairs(examples):
    """Trả về (danh sách câu nguồn, danh sách câu đích) với nhiều kiểu cột."""
    if "translation" in examples:
        src = [ex["en"] for ex in examples["translation"]]
        tgt = [ex["vi"] for ex in examples["translation"]]
    elif "en" in examples and "vi" in examples:
        src, tgt = examples["en"], examples["vi"]
    elif "src" in examples and "tgt" in examples:
        src, tgt = examples["src"], examples["tgt"]
    else:
        raise KeyError(f"Không nhận diện được cột. Các cột hiện có: {list(examples.keys())}")
    return src, tgt


def preprocess_function(examples):
    inputs, targets = extract_pairs(examples)
    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


print("Đang tokenize dữ liệu...")
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# ==============================================================================
# 4. MÔ HÌNH VÀ METRIC
# ==============================================================================
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
metric = evaluate.load("sacrebleu")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    decoded_labels = [[l.strip()] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": round(result["score"], 2)}


# ==============================================================================
# 5. TRAINER (TƯƠNG THÍCH NHIỀU PHIÊN BẢN TRANSFORMERS)
# ==============================================================================
eval_split = "validation" if "validation" in tokenized_datasets else (
    "dev" if "dev" in tokenized_datasets else "test"
)

args_params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in args_params else "evaluation_strategy"

training_kwargs = dict(
    output_dir=CHECKPOINT_DIR,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    eval_steps=EVAL_STEPS,
    save_total_limit=3,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    num_train_epochs=EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)
training_kwargs[eval_key] = "steps"
training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
tok_key = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets[eval_split],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    **{tok_key: tokenizer},
)

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint is not None:
    print(f"\nPhát hiện checkpoint cũ: {last_checkpoint} -> tiếp tục huấn luyện (RESUME)...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\nKhông có checkpoint cũ -> bắt đầu huấn luyện...")
    trainer.train()

final_model_path = os.path.join(CHECKPOINT_DIR, "final_trial_model")
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"\nĐã lưu mô hình tại: {final_model_path}")

# ==============================================================================
# 6. SINH BẢN DỊCH TRÊN TẬP TEST -> preds.txt / refs.txt (DÙNG CHO evaluate.py)
# ==============================================================================
print("\nĐang sinh bản dịch trên tập test...")

test_split = raw_datasets["test"]
test_src, test_ref = extract_pairs(test_split[:])

model = trainer.model
model.eval()
model.to(device)

predictions = []
for i in range(0, len(test_src), GEN_BATCH_SIZE):
    batch = test_src[i:i + GEN_BATCH_SIZE]
    enc = tokenizer(
        batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH
    ).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_length=MAX_LENGTH, num_beams=NUM_BEAMS_TEST)
    predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

preds_path = os.path.join(CHECKPOINT_DIR, "preds.txt")
refs_path = os.path.join(CHECKPOINT_DIR, "refs.txt")

# Mỗi câu 1 dòng: thay xuống dòng bên trong câu bằng dấu cách
with open(preds_path, "w", encoding="utf-8") as f:
    f.write("\n".join(p.strip().replace("\n", " ") for p in predictions) + "\n")
with open(refs_path, "w", encoding="utf-8") as f:
    f.write("\n".join(r.strip().replace("\n", " ") for r in test_ref) + "\n")

print(f"Đã ghi: {preds_path}")
print(f"Đã ghi: {refs_path}")
print(f"\nChấm điểm bằng:\n  python evaluate.py -p \"{preds_path}\" -r \"{refs_path}\"")

Đang chạy trên máy cá nhân -> checkpoint lưu tại thư mục dự án.
Thiết bị: cpu
Thư mục checkpoint: checkpoints_trial

Đang tải tập dữ liệu IWSLT15 en-vi...
Tải thành công từ nhánh parquet của IWSLT/mt_eng_vietnamese
DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 133318
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 1269
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 1269
    })
})
Đang tokenize dữ liệu...


Loading weights: 100%|██████████| 258/258 [00:00<00:00, 18353.95it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



Phát hiện checkpoint cũ: checkpoints_trial\checkpoint-500 -> tiếp tục huấn luyện (RESUME)...


[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].
c:\Users\thach\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 